# PMT A saturation, round 2: anode and dynode-8 pickoff (see `NEXT_STEPS.md`)

**Channels:**
- **Ch1:** PMT A anode;
- **Ch2:** PMT B anode (the light reference);
- **Ch3:** LED drive (the trigger);
- **Ch4:** PMT A **dynode 8 through a 4.7 nF pickoff.** Its pulses are positive.

**Ch4 at two impedances:** every LED point is recorded twice under the same light, first with Ch4 at
**50 Ω** (file `<regime>_<sweep>_...`), then at **1 MΩ** (file `<regime>_<sweep>1M_...`). At 1 MΩ the pickoff
integrates the charge into a slow step. **1 MΩ recordings stop once Ch4 reaches `CH4_1M_STOP_V` (3 V)**, and are
skipped at every brighter LED point, including in the down-sweep. Each new 1 MΩ step is predicted from the
previous one before switching, and the switch is skipped if the prediction is over the limit.

**Light scale:** k (A PE per B PE) comes from the joint occupancy scan (`ratio` cell). The light per point is
k × a smooth fit of B against LED voltage, or A's own PE below the point where B becomes significant.

**Rules:**
- pulse rate < 5 Hz for the saturation scans (4 Hz); 10 Hz for the low-light ratio scan;
- HV is checked, never changed;
- 3 V peak guard on the 50 Ω inputs; burst frames are rejected in the analysis;
- raw frames only are saved, to `runs/saturation_r2/`.

In [ ]:
import numpy as np

REGIME = "short"                         # "short" (100 ns) or "long" (10 us)

# ---- Instruments -----------------------------------------------------------------
SCOPE_RESOURCE = "192.168.0.101"
HV_RESOURCE = "192.168.0.100"
HV_EXPECTED_V = {0: 1550.0, 1: 1550.0}  # HV channel -> expected |V|; checked, never changed
HV_TOLERANCE_V = 2.0
A_CH, B_CH, PULSER_CH, DYN_CH = 1, 2, 3, 4
Q1_PC = {1: 0.9041, 2: 1.09}             # SPE charges; the chain overrides them with a fresh calibration

# ---- LED pulses -------------------------------------------------------------------
PULSE_RATE_HZ = 4.0
assert PULSE_RATE_HZ < 5, "keep the saturation rate below 5 Hz so the bases recharge"
RATIO_RATE_HZ = 10.0                     # low-light joint occupancy scan (user allowed 10 Hz at low light)
assert RATIO_RATE_HZ <= 10
MAX_LED_V = 3.0

# ---- Scope safety and ranging --------------------------------------------------------
MAX_PEAK_V = {A_CH: 3.0, B_CH: 3.0, DYN_CH: 3.0}   # 50 Ohm inputs (rated 5 Vrms; user's 3 V)
CH4_1M_STOP_V = 3.0                      # stop the 1 MOhm Ch4 recordings at this |V| (user, 2026-10-08)
TIMING_1M = False                         # no 1 MOhm in the timing check (it jumps straight to a bright point:
                                          # at 2.0 V / ~700 PE the 1 MOhm trace peaked at ~6 V on 2026-10-08)
PMT_START_SCALE_V_DIV = 0.01
PMT_OFFSET_DIV = 3.0
PROBE_FRAMES = 10
PULSER_SCALE_V_DIV, PULSER_OFFSET_V, TRIG_LEVEL_V = 0.5, -1.5, 0.5

# ---- Regimes (seconds from the trigger) ----------------------------------------------
REGIMES = {
    "short": dict(
        width_s=100e-9, timebase_s_div=100e-9, time_offset_s=0.0, memory_depth="1k", replay_mode="NORM",
        baseline=(-500e-9, -300e-9), signal=(40e-9, 440e-9), control=(-480e-9, -80e-9),
        pre=(-500e-9, 40e-9), pulser_window=(20e-9, 80e-9), plateau=(450e-9, 495e-9),
        occ_signal=(40e-9, 240e-9), occ_control=(-280e-9, -80e-9),
        time_bins=None, early=None, late=None, count_window=None,
        timing_amplitude=2.0, fit_light=(50.0, 300.0),
        # (LED V, pulses): coarse at low light, 10 mV where B becomes usable and A bends
        schedule=[(round(v, 3), 200) for v in np.arange(1.72, 1.921, 0.04)]
                 + [(round(v, 3), 200) for v in np.arange(1.94, 2.141, 0.01)]
                 + [(round(v, 3), 200) for v in np.arange(2.18, 2.421, 0.04)]
                 + [(round(v, 3), 200) for v in np.arange(2.5, 3.001, 0.1)],
    ),
    "long": dict(
        width_s=10e-6, timebase_s_div=2e-6, time_offset_s=5e-6,     # -5 .. +15 us: 5 us before and after
        memory_depth="10k", replay_mode="NORM",   # 500 MS/s acquisition; NORM replay = 1000 pts (20 ns).
                                                  # MAX/RAW replay repeat one record, not the frame (2026-10-08)
        baseline=(-4.8e-6, -0.5e-6), signal=(0.0, 12e-6), control=(-4.8e-6, -0.3e-6),
        pre=(-5e-6, 0.05e-6), pulser_window=(1e-6, 9e-6), plateau=(12.5e-6, 14.9e-6),
        time_bins=np.arange(0.0, 10.01e-6, 1e-6), early=slice(1, 3), late=slice(7, 9),
        count_window=(0.0, 10.5e-6), count_control=(-4.8e-6, -0.3e-6), count_threshold_v=0.005,
        count_max_dt=2e-9,                # photon counting only if ever read at <= 2 ns (not with NORM replay)
        timing_amplitude=1.7, fit_light=(1000.0, 20000.0),
        # Round 3 (reach 2026-10-27): A's charge peaks at ~41 000 PE near 1.9 V; B (~14x dimmer) starts
        # depleting at ~2.2 V (~33 000 PE), so the reference holds to ~2.15 V; 2.4 and 3.0 V show B's own collapse.
        # Round 2: 1.57-1.73 by 0.04, 1.74-1.92 by 0.01, 1.96-2.20 by 0.06, then 2.4-3.0.
        schedule=[(round(v, 3), 150) for v in np.arange(1.60, 1.721, 0.04)]
                 + [(round(v, 3), 200) for v in np.arange(1.76, 2.001, 0.01)]
                 + [(round(v, 3), 150) for v in np.arange(2.04, 2.161, 0.04)]
                 + [(2.4, 150), (3.0, 150)],
    ),
}
# Short pulses wider than 100 ns (round 3, 2026-10-27): the AD3's 100 ns square pulses are badly shaped.
# REGIMES["short"] above is round 2's 100 ns setup; the helpers cell replaces it with short_regime(SHORT_WIDTH_S)
# when the widths differ. Set SHORT_WIDTH_S = 100e-9 to re-analyse round-2 data.
SHORT_WIDTH_S = 250e-9

# Round-3 LED steps for 250 ns pulses (reach pass 2026-10-27: A 580 PE at 1.9 V, knee ~2.0-2.2 V, peak
# flat at ~38 mA above 2.5 V; B linear to 3 V): k_match window (A 400-1000 PE) at ~1.88-1.95 V.
SHORT_SCHEDULE = ([(round(v, 3), 200) for v in np.arange(1.70, 1.821, 0.04)]
                  + [(round(v, 3), 200) for v in np.arange(1.84, 2.401, 0.02)]
                  + [(round(v, 3), 200) for v in np.arange(2.5, 3.001, 0.1)])

def short_regime(w):
    """Short-pulse settings for a pulse of width w (200-300 ns): 200 ns/div, -700 .. +1300 ns at 500 MS/s.

    The light arrives ~80-130 ns after the trigger and lasts w; the windows leave ~200 ns for the tail.
    Control windows have the width of their signal windows and end before the trigger.
    """
    assert 150e-9 <= w <= 400e-9, "short_regime() is laid out for 200-300 ns pulses"
    return dict(
        width_s=w, timebase_s_div=200e-9, time_offset_s=300e-9, memory_depth="1k", replay_mode="NORM",
        baseline=(-700e-9, -450e-9), signal=(40e-9, w + 340e-9), control=(-80e-9 - (w + 300e-9), -80e-9),
        pre=(-700e-9, 40e-9), pulser_window=(50e-9, w - 30e-9), plateau=(w + 350e-9, w + 550e-9),
        occ_signal=(40e-9, w + 190e-9), occ_control=(-80e-9 - (w + 150e-9), -80e-9),
        time_bins=None, early=None, late=None, count_window=None,
        timing_amplitude=1.9, fit_light=REGIMES["short"]["fit_light"],
        schedule=SHORT_SCHEDULE,
    )


DOWN_EVERY = 3                            # down-sweep revisits every 3rd point

# ---- Joint occupancy scan for k (100 ns pulses) ---------------------------------------
# Round 3 (250 ns, new geometry, A/B ~ 25): B's mu is in 0.02-2 at 1.66-1.80 V (A 1-80 PE).
# Round 2 (100 ns): 1.66-1.84 V by 0.03 (500 pulses), 1.86-2.04 V by 0.02 (2000 pulses).
RATIO_SCHEDULE = ([(round(v, 3), 500) for v in np.arange(1.60, 1.681, 0.04)]
                  + [(round(v, 3), 2000) for v in np.arange(1.70, 1.801, 0.02)])
RATIO_A_MAX_PE = 500.0                    # A still linear (round 1: 5 % low at ~1200 PE)
SPE_LOSS_B = 0.005                        # B single PEs below the zero-PE threshold (new B: ~0.5 %)
RATIO_THRESHOLD_Q1 = 0.25                 # zero-PE threshold at 0.25 Q1 above the pedestal; None = 3 MAD sigma.
                                          # 3 sigma was > Q1 for A in the 400 ns window of 250 ns pulses (2026-10-27)

# ---- Analysis --------------------------------------------------------------------------
K_FIXED = None                            # A/B light ratio; None = newest k_*.json from the ratio scan
# Light scale (default "matched"): B smoothed against LED voltage (spline); its scale k_match is fitted to A
# on points with A in MATCH_A_WINDOW (B well measured, A still linear: dynode 8 tracks A within 1 % there).
# Below the window's top A is the light, above it k_match x B_fit; the ratio-scan k is a cross-check.
LIGHT_MODE = "matched"                    # or "ratio_k" (k from the ratio scan, polynomial B fit)
MATCH_A_WINDOW = {"short": (400.0, 1000.0), "long": (5000.0, 30000.0)}
REF_MIN_NPE = 0.05
OUTPUT_DIR = "runs/saturation_r2"
SCOPE_RETRIES = 3

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

from ad3pulse import sat_plots
from ad3pulse.hardware import FrameSet
from ad3pulse.sat_analysis import (b_measure, fit_k_prefix, light_scale, light_scale_matched, load_scan, metric_linearity,
                                   metric_table, ratio_k, summarize_frames, summarize_ratio)

if abs(REGIMES["short"]["width_s"] - SHORT_WIDTH_S) > 1e-12:
    REGIMES["short"] = short_regime(SHORT_WIDTH_S)
cfg = REGIMES[REGIME]
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)


def describe(r50, r1m=None):
    a, b, d = r50["ch"][A_CH], r50["ch"][B_CH], r50["ch"].get(DYN_CH, {})
    text = (f"LED {r50['amplitude_V']:.3f} V: A {a['pe']:9.1f} PE ({a['peak_mA']:6.2f} mA), "
            f"B {b['pe']:8.2f} PE" + (f" / count {b['count']:.2f}" if "count" in b else "")
            + (f", dyn8 {d['q_pC']:9.1f} pC ({d['peak_mA']:.3f} mA)" if "q_pC" in d else ""))
    if r1m is not None:
        text += f", 1 MOhm step {r1m['ch'][DYN_CH]['step_V']:.3f} V"
    if r50["n_rejected"]:
        text += f" [{r50['n_rejected']}/{r50['n_frames']} bursts dropped]"
    return text

In [ ]:
import time
from datetime import datetime

from ad3pulse.acquire import PeakLimitExceeded, ScopeSession, ScopeSettings, autorange, check_peaks
from ad3pulse.hardware import IsegHV
from ad3pulse.instrument import AnalogDiscovery, PulseSpec

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
with IsegHV(HV_RESOURCE) as hv:
    hv_state = {}
    for hv_ch, expected in HV_EXPECTED_V.items():
        hv.channel = hv_ch
        status, v, i = hv.read_status(), hv.read_voltage(), hv.read_current()
        hv_state[hv_ch] = {"V": v, "I_A": i}
        print(f"HV ch{hv_ch}: {v:.2f} V, {i * 1e6:.2f} uA, {', '.join(hv.describe_status(status))}")
        if not status & IsegHV.IS_ON or abs(abs(v) - expected) > HV_TOLERANCE_V:
            raise RuntimeError(f"HV ch{hv_ch} is not on at {expected:g} V")

ad3 = AnalogDiscovery().open()
PULSE = PulseSpec(amplitude=0.5, width=cfg["width_s"], pre_delay=100e-9, post_delay=1e-6)
settings = ScopeSettings(
    pmt_scale_v_div={A_CH: PMT_START_SCALE_V_DIV, B_CH: PMT_START_SCALE_V_DIV, DYN_CH: PMT_START_SCALE_V_DIV},
    polarity={DYN_CH: +1}, impedance={DYN_CH: "FIFTy"}, pulser_channel=PULSER_CH,
    pulser_scale_v_div=PULSER_SCALE_V_DIV, pulser_offset_v=PULSER_OFFSET_V, pmt_offset_div=PMT_OFFSET_DIV,
    timebase_s_div=cfg["timebase_s_div"], time_offset_s=cfg["time_offset_s"], memory_depth=cfg["memory_depth"],
    trig_level_v=TRIG_LEVEL_V)
session = ScopeSession(SCOPE_RESOURCE, settings, ad3, rate_hz=PULSE_RATE_HZ, out_dir=out_dir, stamp=stamp,
                       max_led_v=MAX_LED_V, retries=SCOPE_RETRIES, pulser_window=cfg["pulser_window"],
                       replay_mode=cfg["replay_mode"],
                       run_meta={"regime": REGIME, "width_s": cfg["width_s"], "hv": hv_state, "q1_pC": Q1_PC}).open()
print(f"Scope: {session.scope.idn}; {REGIME}, replay {cfg['replay_mode']}, depth {cfg['memory_depth']}")
for ch in (A_CH, B_CH, DYN_CH):
    print(f"  Ch{ch}: V/div, offset read back = {session.channel_setting(ch)}")

one_m = {"stop_amp": None, "last": None}  # 1 MOhm stop LED amplitude; last (dyn 50 Ohm charge, 1 MOhm step)


def take_point(amp, n, sweep, allow_1m=True):
    """Ch4 at 50 Ohm (all channels), then Ch4 at 1 MOhm (Ch1, Ch2, Ch4) unless stopped or predicted too high.

    The 1 MOhm level is predicted from the previous 1 MOhm point (its peak, which is >= the step)
    scaled by the dynode's 50 Ohm charge; LED steps are small, so the prediction stops 1 MOhm
    before CH4_1M_STOP_V is reached.
    """
    session.set_impedance(DYN_CH, "FIFTy")
    autorange(session, amp, cfg["baseline"], MAX_PEAK_V, n_probe=PROBE_FRAMES)
    fs = session.acquire(amp, n, f"{REGIME}_{sweep}")
    check_peaks(fs, session.settings, cfg["baseline"], MAX_PEAK_V)
    r50 = summarize_frames(fs, cfg, Q1_PC)
    r1m = None
    if not allow_1m or (one_m["stop_amp"] is not None and amp >= one_m["stop_amp"] - 1e-9):
        return r50, None
    q50 = r50["ch"][DYN_CH]["q_pC"]
    if one_m["last"] is not None and one_m["last"][0] > 0 and q50 > 0:
        predicted = one_m["last"][1] * q50 / one_m["last"][0]
        if predicted > 0.9 * CH4_1M_STOP_V:
            one_m["stop_amp"] = amp
            print(f"    1 MOhm skipped from LED {amp:.3f} V on: predicted step {predicted:.2f} V")
            return r50, None
    try:
        session.set_impedance(DYN_CH, "OMEG")
        autorange(session, amp, cfg["baseline"], {DYN_CH: CH4_1M_STOP_V}, n_probe=PROBE_FRAMES, channels=(DYN_CH,))
        fs1 = session.acquire(amp, n, f"{REGIME}_{sweep}1M", channels=(A_CH, B_CH, DYN_CH))
        check_peaks(fs1, session.settings, cfg["baseline"], {**MAX_PEAK_V, DYN_CH: CH4_1M_STOP_V})
        r1m = summarize_frames(fs1, cfg, Q1_PC)
        d1 = r1m["ch"][DYN_CH]
        one_m["last"] = (q50, max(abs(d1["peak_V"]), abs(d1["step_V"])))
    except PeakLimitExceeded as exc:
        if exc.channel != DYN_CH:
            raise
        one_m["stop_amp"] = amp
        print(f"    1 MOhm stopped at LED {amp:.3f} V: {exc}")
    finally:
        session.set_impedance(DYN_CH, "FIFTy")
    return r50, r1m

## Replay readout test (step 2)
Records ~20 dim frames with the long-pulse window at 10k memory depth (500 MS/s), and reads them in NORM and MAX.
**Result on 2026-10-08:** MAX (and RAW) return the same record for every replayed frame, so replay is
limited to NORM's 1000 screen points (20 ns over 20 µs). Kept as a diagnostic.

In [ ]:
from ad3pulse.hardware import RigolScope

L = REGIMES["long"]
session.scope._write(":RUN")
session.scope.configure_timebase(L["timebase_s_div"], L["time_offset_s"], "10k")
ad3.start_pulse_train(PulseSpec(amplitude=0.5, width=10e-6, pre_delay=100e-9, post_delay=1e-6), PULSE_RATE_HZ)
session.rate_hz = PULSE_RATE_HZ
try:
    session.set_amplitude(1.70)
    session.set_scale(A_CH, 0.005); session.set_scale(B_CH, 0.005)
    results = {}
    for mode in ("NORM", "MAX"):
        session.replay_mode = mode
        t0 = time.monotonic()
        fs = session.record(20, (A_CH, B_CH))
        results[mode] = (fs, (time.monotonic() - t0 - 20 / PULSE_RATE_HZ) / 20)
        print(f"{mode}: {fs.time_s.size} points, dt {fs.time_s[1] - fs.time_s[0]:.3g} s, "
              f"readout {results[mode][1] * 1e3:.0f} ms/frame (2 channels)")
finally:
    ad3.stop_output(0)
    session.replay_mode = cfg["replay_mode"]
    session.scope._write(":RUN")
    session.scope.configure_timebase(cfg["timebase_s_div"], cfg["time_offset_s"], cfg["memory_depth"])
fm, _ = results["MAX"]
vb = fm.volts(B_CH)
distinct = len({hash(r.tobytes()) for r in fm.codes[B_CH]})
print(f"MAX: {distinct} distinct frames of {fm.n_frames}; frame-to-frame rms difference "
      f"{np.std(np.diff(vb, axis=0)) * 1e3:.2f} mV")
fn, _ = results["NORM"]
print("compare the two recordings' mean A waveform (different triggers, so only on average):",
      f"NORM mean A {fn.volts(A_CH).mean() * 1e3:.3f} mV, MAX mean A {fm.volts(A_CH).mean() * 1e3:.3f} mV")

## Joint occupancy scan for k (step 3)
100 ns pulses at `RATIO_RATE_HZ`, recording Ch1 and Ch2, auto-ranged per point. For each point it computes both
PMTs' zero-PE μ and mean PE. k = A PE / B μ is taken where A is still linear and B's μ is significant.

In [ ]:
S = REGIMES["short"]
session.scope._write(":RUN")
session.scope.configure_timebase(S["timebase_s_div"], S["time_offset_s"], S["memory_depth"])
session.rate_hz = RATIO_RATE_HZ
ad3.start_pulse_train(PulseSpec(amplitude=0.5, width=S["width_s"], pre_delay=100e-9, post_delay=1e-6), RATIO_RATE_HZ)
ratio_rows = []
try:
    for amp, n in RATIO_SCHEDULE:
        t0 = time.monotonic()
        autorange(session, amp, S["baseline"], MAX_PEAK_V, n_probe=PROBE_FRAMES, channels=(A_CH, B_CH))
        fs = session.acquire(amp, n, "ratio", channels=(A_CH, B_CH))
        r = summarize_ratio(fs, S, Q1_PC, threshold_q1=RATIO_THRESHOLD_Q1)
        ratio_rows.append(r)
        print(f"LED {amp:.3f} V ({n} pulses, {time.monotonic() - t0:.0f} s): mu_A {r[1]['mu']:.3f}±{r[1]['mu_err']:.3f}, "
              f"A {r[1]['pe']:.2f} PE, mu_B {r[2]['mu']:.4f}±{r[2]['mu_err']:.4f}, B {r[2]['pe']:.4f} PE", flush=True)
finally:
    ad3.stop_output(0)
    session.rate_hz = PULSE_RATE_HZ
    session.scope._write(":RUN")
    session.scope.configure_timebase(cfg["timebase_s_div"], cfg["time_offset_s"], cfg["memory_depth"])

In [ ]:
S = REGIMES["short"]
if "ratio_rows" not in globals() or not ratio_rows:
    files = sorted(out_dir.glob("ratio_*.npz"))
    run = max("_".join(f.stem.split("_")[-2:]) for f in files)
    ratio_rows = [summarize_ratio(FrameSet.load(f), S, Q1_PC, threshold_q1=RATIO_THRESHOLD_Q1) for f in files if f.stem.endswith(run)]
    ratio_rows.sort(key=lambda r: r["amplitude_V"])
    ratio_stamp = run
else:
    ratio_stamp = stamp
kres = ratio_k(ratio_rows, a_max_pe=RATIO_A_MAX_PE, spe_loss_b=SPE_LOSS_B)
print(f"k = {kres['k_mean']:.1f} ± {kres['k_mean_err']:.1f} from {kres['use'].sum()} points "
      f"(chi2/dof {kres['chi2_dof']:.2f})")
for r in ratio_rows:
    if 0 < r[1]["mu"] < 3 and r[1]["mu"] > 3 * r[1]["mu_err"]:
        print(f"  A check at {r['amplitude_V']:.3f} V: charge/Q1 {r[1]['pe']:.3f} vs zero-PE mu {r[1]['mu']:.3f}")
    if 0 < r[2]["mu"] < 3 and r[2]["mu"] > 3 * r[2]["mu_err"]:
        print(f"  B check at {r['amplitude_V']:.3f} V: charge/Q1 {r[2]['pe']:.3f} vs zero-PE mu {r[2]['mu']:.3f}")
(out_dir / f"k_{ratio_stamp}.json").write_text(json.dumps(
    {"k": kres["k_mean"], "k_err": kres["k_mean_err"], "chi2_dof": kres["chi2_dof"], "q1_pC": Q1_PC,
     "n_points": int(kres["use"].sum()), "a_max_pe": RATIO_A_MAX_PE, "spe_loss_b": SPE_LOSS_B, "threshold_q1": RATIO_THRESHOLD_Q1}, indent=1))
fig = sat_plots.plot_ratio_scan(ratio_rows, kres, f"joint occupancy scan {ratio_stamp} ({S['width_s'] * 1e9:g} ns, {RATIO_RATE_HZ:g} Hz)")
fig.savefig(out_dir / f"ratio_{ratio_stamp}.png", dpi=140)
plt.show()

## Timing check

In [ ]:
ad3.start_pulse_train(PULSE, PULSE_RATE_HZ)
try:
    r50, r1m = take_point(cfg["timing_amplitude"], 20, f"timing", allow_1m=TIMING_1M)
    print(describe(r50, r1m))
except PeakLimitExceeded as exc:
    r50 = None
    print(f"timing check skipped, PEAK GUARD: {exc}")
finally:
    ad3.stop_output(0)
if r50 is not None:
    fig, ax = plt.subplots(figsize=(9, 4))
    ts = 1e6 if REGIME == "long" else 1e9
    for ch, name in ((A_CH, "A anode (-V)"), (B_CH, "B anode (-V)"), (DYN_CH, "A dynode 8 (+V)")):
        mv = r50["ch"][ch]["mean_v"] * (1 if ch == DYN_CH else -1)
        ax.plot(r50["t"] * ts, mv / max(np.max(mv), 1e-12), lw=1, label=name)
    for (lo, hi), col in ((cfg["signal"], "C2"), (cfg["baseline"], "C0"), (cfg["plateau"], "0.6")):
        ax.axvspan(lo * ts, hi * ts, color=col, alpha=0.12)
    ax.set_xlabel("time from trigger (µs)" if REGIME == "long" else "time from trigger (ns)")
    ax.set_ylabel("mean / peak"); ax.grid(alpha=.3); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(out_dir / f"{REGIME}_timing_{stamp}.png", dpi=140); plt.show()

## Saturation scan (step 4)
An up-sweep over `cfg["schedule"]`, then a down-sweep over every `DOWN_EVERY`-th point. Each point is recorded
with Ch4 at 50 Ω and then at 1 MΩ (see `take_point`).

In [ ]:
schedule = cfg["schedule"]
one_m.update(stop_amp=None, last=None)
ad3.start_pulse_train(PULSE, PULSE_RATE_HZ)
try:
    top = len(schedule)
    for k, (amp, n) in enumerate(schedule):
        t0 = time.monotonic()
        try:
            r50, r1m = take_point(amp, n, "up")
        except PeakLimitExceeded as exc:
            print(f"LED {amp:.3f} V: PEAK GUARD: {exc}; ending the up-sweep")
            top = k
            break
        print("up  ", describe(r50, r1m), f"[{time.monotonic() - t0:.0f} s]", flush=True)
    for amp, n in schedule[:top][-2::-DOWN_EVERY]:
        r50, r1m = take_point(amp, n, "down")
        print("down", describe(r50, r1m), flush=True)
finally:
    ad3.stop_output(0)

## Load a saved scan (no hardware)

In [ ]:
LOAD_STAMP = None
points, run = load_scan(out_dir, REGIME, cfg, Q1_PC, LOAD_STAMP)
stamp = f"analysis_{run}"
print(f"{REGIME} run {run}: {len(points)} points, {sum(p.r1m is not None for p in points)} with 1 MOhm")

## Analysis
- **Light scale:** k from the ratio scan, times the smooth B fit. For long pulses B is photon-counted where the
  sampling allows. A long-pulse k fitted at the dim end is printed as a cross-check.
- **Metrics:** each metric is scaled to the light on `cfg["fit_light"]`. The metrics are A's anode charge and
  peak, dynode 8 at 50 Ω (charge and peak), and dynode 8 at 1 MΩ (step).
- **Plots:** response against LED voltage, A against B with the light curve, linearity of all metrics, Ch1 and
  Ch4 pulse shapes, and shape metrics.

In [ ]:
if "points" not in globals() or not points or points[0].r50["label"].split("_")[0] != REGIME:
    points, run = load_scan(out_dir, REGIME, cfg, Q1_PC)
    stamp = f"analysis_{run}"
k, k_err = (K_FIXED, 0.0) if isinstance(K_FIXED, (int, float)) else (None, None)
if k is None:
    kfiles = sorted(out_dir.glob("k_*.json"))
    if kfiles:
        kj = json.loads(kfiles[-1].read_text())
        k, k_err = kj["k"], kj["k_err"]
        print(f"k from {kfiles[-1].name}: {k:.1f} ± {k_err:.1f}")
b, b_err, counted = b_measure(points)
if REGIME == "long":
    try:
        k_long, k_long_err, idx = fit_k_prefix(points, b, b_err, REF_MIN_NPE, 3, min_late_early=0.5)
        print(f"cross-check: long-pulse k at the dim end = {k_long:.1f} ± {k_long_err:.1f} "
              f"(B {'counted' if counted[idx].all() else 'charge'})")
    except ValueError as exc:
        print("long-pulse k cross-check failed:", exc)
if k is None:
    k, k_err, _ = fit_k_prefix(points, b, b_err, REF_MIN_NPE, 3,
                               min_late_early=0.5 if REGIME == "long" else None)
    print(f"no ratio-scan k: fitted k = {k:.1f} ± {k_err:.1f}")
if LIGHT_MODE == "matched":
    ls = light_scale_matched(points, b, b_err, MATCH_A_WINDOW[REGIME], b_min=REF_MIN_NPE, k_ref=k)
    print(f"matched light scale: k_match = {ls['k']:.1f} ± {ls['k_err']:.1f} from {ls['window'].sum()} points "
          f"with A in {MATCH_A_WINDOW[REGIME]} PE (ratio-scan k = {k:.1f} ± {k_err:.1f})")
    k, k_err = ls["k"], ls["k_err"]
    fit = ls["window"]
else:
    ls = light_scale(points, k, k_err, b_min=REF_MIN_NPE, b=b, b_err=b_err)
    fit = (ls["light"] >= cfg["fit_light"][0]) & (ls["light"] <= cfg["fit_light"][1])
metrics = metric_table(points)
lins = {name: metric_linearity(v, e, ls["light"], ls["light_err"], fit)
        for name, (v, e) in metrics.items() if not name.startswith("B ")}
print(f"light scale: B used from LED {ls['v_on']:.3f} V; metric scales from {fit.sum()} points")
summary = {}
for name, lin in lins.items():
    c5, c10 = lin["cross"][0.05], lin["cross"][0.10]
    summary[name] = {"5%": c5, "10%": c10, "n_fit": lin["n_fit"]}
    print(f"  {name:32s}: 5 % low at {c5 if c5 is None else f'{c5:.4g}'} PE, "
          f"10 % low at {c10 if c10 is None else f'{c10:.4g}'} PE (scale from {lin['n_fit']} points)")
(out_dir / f"{REGIME}_summary_{run}.json").write_text(json.dumps(
    {"k": k, "k_err": k_err, "v_on": ls["v_on"], "crossings_true_light_PE": summary}, indent=1, default=str))

figs = {
    "vs_led": sat_plots.plot_vs_led(points, metrics, REGIME),
    "a_vs_b": sat_plots.plot_a_vs_b(points, ls, k, REGIME),
    "linearity": sat_plots.plot_linearity(
        points, ls, lins, REGIME,
        ("anode peak current (mA)", metrics["A anode peak (mA)"][0]) if REGIME == "short"
        else ("expected anode charge, light × Q1 (nC)", ls["light"] * Q1_PC[A_CH] / 1e3)),
    "shapes": sat_plots.plot_shapes_ch1_ch4(points, ls["light"], cfg, REGIME),
    "shape_metrics": sat_plots.plot_shape_metrics(points, ls["light"], REGIME),
}
for name, fig in figs.items():
    if fig is not None:
        fig.savefig(out_dir / f"{REGIME}_{name}_{run}.png", dpi=140)
plt.show()

## Close

In [ ]:
ad3.stop_output(0)
ad3.close()
session.close()

## Emergency: LED pulses off (HV untouched)

In [ ]:
try:
    ad3.stop_output(0)
    ad3.close()
except NameError:
    from ad3pulse.instrument import AnalogDiscovery
    with AnalogDiscovery() as _dev:
        _dev.stop_output(0)
print("AD3 output stopped")